In [ ]:
%load_ext autoreload
%autoreload 2
import waffles
import numpy as np
import plotly.subplots as psu
import os
from tqdm import tqdm

from waffles.data_classes.Waveform import Waveform
from waffles.data_classes.WaveformSet import WaveformSet
from waffles.data_classes.ChannelWsGrid import ChannelWsGrid
from waffles.utils.selector_waveforms import WaveformSelector
from waffles.np02_utils.AutoMap import ordered_modules_cathode, ordered_modules_membrane, ordered_modules_pmt, dict_module_to_uniqch, dict_uniqch_to_module, strUch, getModuleName
from waffles.np02_utils.PlotUtils import runBasicWfAnaNP02, wfset_remove_bad_baselines, plot_detectors, plot_waveforms
from waffles.np02_utils.load_utils import open_processed, ch_read_calib
from waffles.utils.numerical_utils import average_wf_ch, compute_mpv_waveforms
from utils import save_templates

In [ ]:
dettype = "membrane"
dettype = "cathode"
#dettype = "pmt"

datadir = f"/eos/experiment/neutplatform/protodune/experiments/ProtoDUNE-VD/commissioning/"
endpoint = 106 if dettype == "cathode" else 107 if dettype == "membrane" else 110
listmods = ordered_modules_cathode if endpoint == 106 else ordered_modules_membrane if endpoint == 107 else ordered_modules_pmt

n = len(listmods)
group1 = listmods[:n//4]
group2 = listmods[n//4:n//2]
group3 = listmods[n//2:2*(n//3+1)]
group4 = listmods[2*(n//3+1):]

groupLow = group1+group2
groupHig = group3+group4
groupall = group1+group2+group3+group4

try:
    tmpep = list(wfset_full.get_set_of_endpoints())
    if endpoint != tmpep[0]: # recreate the wfset, only one endpoint at a time
        del wfset_full
        wfset_full = None
except:
    wfset_full=None

In [ ]:
run_to_module = {

    "membrane" : {
        40801 : ["M3(1)", "M3(2)"], # Mask 8, int 2250
        40989 : ["M4(1)", "M4(2)"], # Mask 8, int 1900 
        40808 : ["M6(1)", "M6(2)"], # Mask 16, int 4000
        42320 : ["M5(1)", "M5(2)"], # Mask 2, int 4000   
        43229 : ["M7(1)", "M7(2)"], # Mask 4, int 1400 - if it doesn't work try 43230
        42321 : ["M8(1)", "M8(2)"], # Mask 4, int 4000
    },
    "cathode" : { 
       42228 : ["C1(1)", "C1(2)"], # Mask 8, int 4000 - fibers swapped - provare mask 4
       40808 : ["C4(1)", "C4(2)", "C5(1)", "C5(2)", "C7(1)", "C7(2)", "C8(1)", "C8(2)"], # Mask 16, int 4000
       41519 : ["C2(1)", "C2(2)"], # Mask 8, int 3000 
       41536 : ["C3(1)", "C3(2)"], # Mask 1, 2800 
       40806 : ["C6(1)", "C6(2)"], # Mask 8, int 4000  
    },

    "pmt" : {
      43282 : ordered_modules_pmt 
        
    }

}
run_to_module = run_to_module[dettype]

run_to_unich = { r: [ dict_module_to_uniqch[m].channel for m in modules ] for r, modules in run_to_module.items() }
channels = [ x for v in run_to_unich.values() for x in v]

In [ ]:
import copy
import time
def select_channels(waveform: Waveform, channels: list) -> bool:
    if waveform.channel not in channels:
        return False
    if np.any(waveform.adcs[0:500] > 16000):
        return False
    if np.any(waveform.adcs[0:500] < 100 ):
        return False
    return True
def remove_channel_when_run_change(waveform: Waveform, channels_to_remove = []) -> bool:
    if waveform.channel in channels_to_remove:
        return False
    return True
    
def create_wfset(run_to_unich, endpoint, wfset_alread_there=None, nwf_per_run=None):
    nwaveforms = 1000
    wfset_full: WaveformSet = wfset_alread_there

    if wfset_full is not None:
        for run in wfset_full.runs:
            if run not in run_to_unich.keys():
                if len(wfset_full.runs) != 1:
                    print(f"Removing unused run {run}")
                    wfset_full = WaveformSet.from_filtered_WaveformSet(
                        wfset_full, remove_channel_when_run_change,
                        wfset_full.available_channels[run][endpoint], show_progress=True
                    )
                else:
                    wfset_full = None

    if nwf_per_run is None:
        nwf_per_run = {}

    for run, channels in run_to_unich.items():
        if wfset_full is not None:
            redo = True

            if run in wfset_full.runs:
                channels_match = list(wfset_full.available_channels[run][endpoint]) == sorted(channels)
                nwf_match = nwf_per_run.get(run) == nwaveforms

                if channels_match and nwf_match:
                    print(f"Run {run} already there :)")
                    redo = False
                else:
                    if not channels_match:
                        print(f"Run {run}: channels changed, reprocessing...")
                    else:
                        print(f"Run {run}: nwaveforms changed, reprocessing...")

            if redo:
                try:
                    wfset_full = WaveformSet.from_filtered_WaveformSet(
                        wfset_full, remove_channel_when_run_change,
                        channels_to_remove=channels, show_progress=True
                    )
                except:
                    wfset_full = None
            else:
                continue

        wfset = open_processed(run, dettype, datadir, channels, [endpoint],
                               nwaveforms=nwaveforms, verbose=True, mergefiles=True)
        if wfset_full is None:
            wfset_full = copy.deepcopy(wfset)
        else:
            wfset_full.merge(copy.deepcopy(wfset))
        
        nwf_per_run[run] = nwaveforms  # save it here!
        print(f"Loaded run {run}")

    return wfset_full, nwf_per_run


# call it like this, keeping nwf_per_run across calls
try:
    nwf_per_run
except NameError:
    nwf_per_run = {}

start = time.time()
wfset_full, nwf_per_run = create_wfset(run_to_unich, endpoint, wfset_full, nwf_per_run)
end = time.time()
print(end - start)

In [ ]:
runBasicWfAnaNP02(wfset_full, int_ll=250, int_ul=280, amp_ll=100, amp_ul=260, threshold=25, onlyoptimal=False, configyaml="")

In [ ]:
wfset_optimal = wfset_remove_bad_baselines(wfset_full)
wfset_optimal

In [ ]:
argsheat = dict(
    mode="heatmap",
    analysis_label="std",
    adc_range_above_baseline=2000,
    adc_range_below_baseline=-150,
    adc_bins=125,
    time_bins=wfset_full.points_per_wf//2,
    filtering=16,
    share_y_scale=True,
    share_x_scale=True,
    wfs_per_axes=5000,
    zlog=True,
    width=1550,
    higth=1300,
    showplots=True
)

#detector = groupLow
#detector = groupHig
detector = groupall

#detector=["C1(1)","C1(2)"]

plot_detectors(wfset_optimal, detector, **argsheat)


In [ ]:
cutyaml = 'cuts.yaml'
extractor = WaveformSelector(cutyaml)
wfset_clean = WaveformSet.from_filtered_WaveformSet(wfset_optimal, extractor.applycuts, show_progress=True)
print(f"Original waveforms: {len(wfset_optimal.waveforms)}, after cut: {len(wfset_clean.waveforms)}")

In [ ]:
def generate_averages(wfset, template_type, maxfev=8000):
    wfsetch = ChannelWsGrid.clusterize_waveform_set(wfset)
    ret = {}
    peaks = {}
    for ep, wfch in wfsetch.items():
        ret[ep] = ret.get(ep, {})
        for ch, wfs in wfch.items():
            if template_type == "average":
                average = average_wf_ch(wfs)
            elif template_type == "MPV":
                fit_info = compute_mpv_waveforms(wfs, maxfev=maxfev)
                average = fit_info["mpv"]
            else:
                raise Exception("Method not implemented...") 
                
            ret[ep][ch] = average
            peaks[(ep, ch)] = np.max(average)

    return ret, peaks

In [ ]:
#template_type = "MPV"
template_type = "average"
averages, peaks_avg = generate_averages(wfset_clean, template_type)

In [ ]:
argsheat['adc_range_above_baseline']=2000
argsheat['adc_range_below_baseline']=-200
argsheat['adc_bins']=125
argsheat['wfs_per_axes']=None
argsheat['return_fig'] = True
figs = plot_detectors(wfset_clean, detector, **argsheat)
fig, rows, cols, title, g = figs[0]
plot_waveforms(fig, g, averages, line_color='blue')
fig.show()

In [ ]:
calibration_file = 'np02-config-v4.0.0.csv'
calibration_data = ch_read_calib(calibration_file)
wfsetch = ChannelWsGrid.clusterize_waveform_set(wfset_clean)
wfsetch = { ep: { ch: w for ch, w in sorted(wfsch.items(), key=lambda x: getModuleName(ep,x[0])) } for ep, wfsch in wfsetch.items() }

In [ ]:
avg_norm = {}
for ep, wfsch in wfsetch.items():
    avg_norm[ep] = avg_norm.get(ep, {})
    for ch, wfs in wfsch.items():
        avg = averages[ep][ch]
        peak = peaks_avg[(ep, ch)]
        if peak == 0 or np.isnan(peak):
            print(f"Zero or NaN peak for channel {ch}")
            continue

        if ep in calibration_data and ch in calibration_data[ep]:
            spe_amp = calibration_data[ep][ch]["SpeAmpl"]
        else:
            print(f"No calibration data found for {getModuleName(ep,ch)}: {ep}-{ch}")
            spe_amp = 1
            peak = 1

        avg_norm[ep][ch] = avg * (spe_amp / peak )


In [ ]:
rows = g.ch_map.rows
cols = g.ch_map.columns
fig_simple = psu.make_subplots(rows=rows, cols=cols, subplot_titles=[" "] * (rows * cols))
plot_waveforms(fig_simple, g, waveforms=averages)

fig_simple.update_layout(template="plotly_white", width=argsheat['width'], height=argsheat['higth'], showlegend=False)
fig_simple.update_annotations( font=dict(size=14), align="center", )

fig_simple.show()

In [ ]:
template_name = 'test'
save_templates(wfsetch, template_name=template_name, cutyaml=cutyaml, detector=detector, templates=avg_norm, peaks=peaks_avg, dry_run=False)